# Práctica 4: Listas, Mutabilidad y Técnicas de Iteración

**Informática · Grado en Ingeniería Aeroespacial · UCLM**
Curso 2026/27

> **Instrucciones:** Guarda una copia de este notebook en tu Google Drive (**Archivo > Guardar una copia en Drive**) y resuelve los ejercicios en las celdas indicadas. Cada ejercicio cuenta con celdas para tu solución y celdas de comprobación para verificar su correcto funcionamiento.


## Listas en Ingeniería Aeroespacial

Las listas son la estructura básica para guardar series temporales de telemetría, mallas de presión de un ensayo en túnel de viento o colas de tráfico aéreo. En esta práctica trabajamos los Temas 4.1 y 4.2: indexación y slicing, métodos y ordenación, aliasing y copias, y las técnicas de iteración con comprehensions.


## Parte 1: Indexación, slicing y ordenación (Tema 4.1)

Una caja negra (FDR) guarda la aceleración vertical como una lista de muestras: el analista extrae tramos concretos con slicing y ordena datos sin destruir la serie original. Practicamos índices positivos y negativos, los tres parámetros del slice, los métodos esenciales y la diferencia entre `.sort()` y `sorted()`.


### Ejercicio 1: Tramos de una caja negra (indexación y slicing)

El registrador de datos de vuelo (FDR) de un avión de transporte muestrea la aceleración vertical $n_z$ (en $g$) durante un encuentro con turbulencia. La serie registrada (12 muestras a 4 Hz) es:

```python
fdr_nz = [0.98, 1.02, 1.05, 1.12, 2.30, 2.45, -0.35, -0.80, 0.60, 1.05, 1.10, 0.95]
```

**Tareas:**
1. Define la lista `fdr_nz` con las 12 muestras anteriores.
2. Define la función `extraer_tramos(fdr_nz, inicio_turb=4, fin_turb=8, n_cola=3)` que devuelva la tupla `(cabecera, turbulencia, cola, invertida)`:
   * `cabecera = fdr_nz[:6]`: las **primeras 6** muestras del vuelo.
   * `turbulencia = fdr_nz[inicio_turb:fin_turb]`: recuerda que el límite superior del slice **no se incluye**.
   * `cola = fdr_nz[-n_cola:]`: las últimas muestras, con índice negativo.
   * `invertida = fdr_nz[::-1]`: la serie completa en orden inverso (paso negativo).
3. Con el tramo de turbulencia, obtén el pico máximo y el mínimo con `max()` y `min()`, y la media de la serie completa con `sum()` y `len()`.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: cabecera = [0.98, 1.02, 1.05, 1.12, 2.3, 2.45] · turbulencia = [2.3, 2.45, -0.35, -0.8] ·
# cola = [1.05, 1.1, 0.95] · picos en la turbulencia: 2.45 g y -0.8 g · media global = 0.9558 g
cabecera, turbulencia, cola, invertida = extraer_tramos(fdr_nz)
print("Cabecera (primeras 6):", cabecera)
print("Tramo de turbulencia:  ", turbulencia)
print("Cola (últimas 3):      ", cola)
print("Serie invertida:       ", invertida)
print(f"Picos de la turbulencia: máx {max(turbulencia)} g · mín {min(turbulencia)} g")
print(f"Media de la serie completa: {sum(fdr_nz) / len(fdr_nz):.4f} g")


---
### Ejercicio 2: La cola de aproximación (métodos y ordenación)

La torre de control de la pista 27 gestiona la cola de aproximación y las altitudes de crucero de los vuelos en espera:

```python
cola = ["AEA2812", "IBE6014", "RYR7720", "VLG1105", "IBE6014"]
altitudes = [11200, 4500, 8900, 6100, 3600]
```

**Tareas:**
1. Define la función `gestionar_aproximaciones(cola, emergencia, desviado, nuevo)` que, en este orden:
   * Extraiga con `pop(0)` al primer vuelo de la cola, que es el que aterriza (guárdalo en una variable).
   * Inserte `emergencia` al **frente** de la cola con `insert(0, emergencia)`.
   * Elimine con `remove(desviado)` el vuelo rerutado a un aeropuerto alternativo (borra la **primera** aparición).
   * Añada `nuevo` al final con `append()`.
   * Devuelva la tupla `(aterrizado, cola)`.
2. Define la función `ordenar_niveles(altitudes)` que devuelva una **nueva lista** ordenada de **mayor a menor**, sin modificar la original. Pista: `sorted()` con `reverse=True`.
3. Comprueba en una celda auxiliar qué devuelve `altitudes.sort()` al imprimirlo: por eso nunca debes escribir `altitudes = altitudes.sort()`.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: aterriza AEA2812 · cola final = ['AEA1188', 'IBE6014', 'VLG1105', 'IBE6014', 'IBK2831'] ·
# 'IBE6014' aparece 2 veces (primera en el índice 1) · niveles ordenados = [11200, 8900, 6100, 4500, 3600] ·
# la lista original queda intacta
cola = ["AEA2812", "IBE6014", "RYR7720", "VLG1105", "IBE6014"]
altitudes = [11200, 4500, 8900, 6100, 3600]
aterrizado, cola_final = gestionar_aproximaciones(cola, emergencia="AEA1188", desviado="RYR7720", nuevo="IBK2831")
print("Vuelo aterrizado:", aterrizado)
print("Cola final:      ", cola_final)
print("'IBE6014' aparece", cola_final.count("IBE6014"), "veces; primera aparición en el índice", cola_final.index("IBE6014"))
print("Niveles ordenados:", ordenar_niveles(altitudes))
print("Lista original:   ", altitudes)


## Parte 2: Mutabilidad, aliasing y copias (Tema 4.2)

Una asignación `B = A` no copia nada: ambos nombres apuntan al mismo objeto en memoria. Con un registro de fallos de motor comprobamos el aliasing, hasta dónde protege una copia superficial y cuándo hace falta `copy.deepcopy()`.


### Ejercicio 3: El espejismo (aliasing, copia superficial y copia profunda)

El sistema de mantenimiento clona el registro de fallos de un motor para analizarlo en tierra. Realiza las tareas **en orden**, imprimiendo tras cada paso lo que se pide:

```python
registro = ["MOT-1", ["temp_aceite", 142], ["vibracion", 2.1]]
```

**Tareas:**
1. Crea la lista `registro` y haz `alias = registro`. Comprueba que `alias is registro` y `alias == registro` valen `True`: son el mismo objeto.
2. Crea una copia superficial `copia_sup = registro[:]` (o `registro.copy()`) y cambia su elemento de primer nivel: `copia_sup[0] = "MOT-2"`. Comprueba `registro[0]`: el primer nivel **sí** queda protegido.
3. Modifica ahora una sublista de la copia: `copia_sup[1][1] = 200`. Comprueba `registro[1]`: las sublistas **se comparten** aunque hayas copiado.
4. Crea una copia profunda `copia_prof = copy.deepcopy(registro)` (importa el módulo `copy`), cambia `copia_prof[2][1] = 9.9` y comprueba que `registro` no se ve afectado.
5. Define dos listas independientes `p = [1, 2, 3]` y `q = [1, 2, 3]` y comprueba que `p == q` es `True` pero `p is q` es `False`: mismo valor, objetos distintos.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: alias is registro → True · tras la copia superficial, registro[0] = MOT-1 pero
# registro[1] = ['temp_aceite', 200] (sublista compartida) · tras deepcopy, registro[2] = ['vibracion', 2.1] ·
# p == q → True y p is q → False
import copy
print("alias is registro:", alias is registro)
print("Primer nivel tras copia superficial:", registro[0])
print("Sublista tras copia superficial:    ", registro[1])
print("Sublista tras deepcopy:             ", registro[2])
print("p == q:", p == q, "· p is q:", p is q)


---
### Ejercicio 4: La trampa de borrar elementos mientras se itera

Una radiosonda suelta lecturas erróneas (`None` o valores negativos) que hay que limpiar. Si borramos de la lista **dentro** de un `for x in lecturas:` que la está recorriendo, el iterador se desincroniza con los índices y se saltan elementos.

**Tareas:**
1. Comprueba primero el efecto: con `datos = [12.5, None, -3.2, 14.1, None, None, 18.9, -1.0]`, borra dentro de un `for x in datos:` los valores no válidos (`x is None or x < 0`) e imprime la lista al terminar: quedan elementos sin eliminar.
2. Define la función `limpiar_sensores(lecturas)` que elimine **todos** los valores no válidos de forma segura, iterando sobre una copia (`for x in lecturas[:]`) o construyendo una lista nueva.
3. La función debe modificar la lista original *in situ* y no devolver nada.

*Pista:* `None` no es comparable con `<` (lanza `TypeError`); comprueba siempre `x is None` **antes** de evaluar `x < 0`.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: datos_sucios queda como [12.5, 14.1, 18.9], sin None ni negativos
datos_sucios = [12.5, None, -3.2, 14.1, None, None, 18.9, -1.0]
limpiar_sensores(datos_sucios)
print("Datos limpios:", datos_sucios)


## Parte 3: Técnicas de iteración y comprehensions (Temas 4.1–4.2)

Recorrer una lista por índice permite comparar cada elemento con sus vecinos; una list comprehension condensa el patrón acumulador (filtrar y transformar) en una sola línea.


### Ejercicio 5: Iterar por índice (detección de picos de vibración)

Para detectar anomalías transitorias en la estructura de un ala comparamos cada lectura de vibración con sus vecinas inmediata anterior y posterior. Una lectura en el índice $i$ es un **pico local** si:

$$v[i] > v[i-1] \quad \text{y} \quad v[i] > v[i+1] \quad \text{y} \quad v[i] > \text{umbral}$$

**Tareas:**
1. Define la función `detectar_picos_vibracion(sensores_vibracion, umbral)`.
2. Itera por índice con `range(1, len(v) - 1)`: ni la primera ni la última lectura pueden ser picos porque no tienen dos vecinas.
3. Para cada índice que cumpla las tres condiciones, añade con `append()` la tupla `(i, v[i])` a una lista resultado y devuélvela al terminar.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: [(2, 0.88), (5, 0.95)]
vibraciones = [0.12, 0.15, 0.88, 0.20, 0.18, 0.95, 0.30, 0.22]
picos = detectar_picos_vibracion(vibraciones, umbral=0.50)
print("Picos detectados (índice, valor):", picos)


---
### Ejercicio 6: Filtrar y convertir con una comprehension (radiosonda)

Una radiosonda devuelve temperaturas en grados Celsius y algunas muestras caen fuera del rango de validez del sensor. El patrón «filtrar y transformar» se escribe en una sola línea con una list comprehension (Tema 4.2).

**Tareas:**
1. Define la función `filtrar_y_convertir_temperaturas(temps_celsius, t_min_c=-60, t_max_c=50)`.
2. Con **una única comprensión de lista**, descarta las temperaturas fuera del rango $[t_{\text{min}}, t_{\text{max}}]$ y convierte cada temperatura válida a kelvin ($K = ^\circ\text{C} + 273.15$).
3. Devuelve la lista resultante.

*Nota:* al imprimir la lista directamente puede aparecer `223.14999999999998` en lugar de `223.15`: es el error de coma flotante del Tema 3.2. La celda de comprobación muestra los valores redondeados a dos decimales.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: [223.15, 253.15, 273.15, 288.15, 301.15, 315.15] (descartadas -75, 85 y 120)
muestras_c = [-75, -50, -20, 0, 15, 28, 42, 85, 120]
kelvin_validas = filtrar_y_convertir_temperaturas(muestras_c)
print("Temperaturas en kelvin (redondeadas):", [round(k, 2) for k in kelvin_validas])


---
### Ejercicio 7 (ampliación): Malla de presiones en túnel de viento (matrices)

Ejercicio opcional, para casa o para quien termine antes. En un ensayo en túnel de viento el ala se instrumenta con una malla de tomas de presión: cada valor es un coeficiente de presiones $C_p$ (negativo = succión); cada **fila** es una estación a lo largo de la envergadura y cada **columna**, una toma en la cuerda del perfil:

```python
cp_malla = [
    [-1.25, -0.80, -0.35],
    [-1.40, -0.95, -0.40],
    [-1.15, -0.70, -0.30],
    [-1.55, -1.05, -0.45]
]
```

**Tareas:**
1. Imprime `cp_malla[2][1]`: el primer índice selecciona la **fila** y el segundo la **columna**.
2. Define `succiones_por_estacion(cp_malla)` que devuelva, con una comprehension, el $C_p$ de máxima succión (el mínimo) de cada fila: patrón `[min(fila) for fila in cp_malla]`.
3. Define `succiones_por_toma(cp_malla)` que haga lo mismo por columna: recorre los índices de columna con `range(len(cp_malla[0]))` y extrae cada columna con una comprehension anidada, el mismo patrón de la transpuesta del Tema 4.2.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación
# Valores esperados: cp_malla[2][1] = -0.7 · por estación = [-1.25, -1.4, -1.15, -1.55] ·
# por toma = [-1.55, -1.05, -0.45]
print("Cp en la fila 2, columna 1:", cp_malla[2][1])
print("Máximas succiones por estación (filas):", succiones_por_estacion(cp_malla))
print("Máximas succiones por toma (columnas): ", succiones_por_toma(cp_malla))


---

**Departamento de Tecnologías y Sistemas de Información**
Escuela de Ingeniería Industrial y Aeroespacial de Toledo — UCLM
